<div style="text-align:center; font-family: sans-serif; padding:40px; background:linear-gradient(135deg,#1a1a2e,#16213e); color:white; border-radius:12px;">
  <h1>Robot de Inventario Amazon</h1>
  <h2 style="color:#a0c4ff;">Resolucion mediante Busqueda Heuristica A*</h2>
  <hr style="border-color:#a0c4ff55; width:60%"/>
  <p><b>Asignatura:</b> Razonamiento y Planificacion Automatica</p>
  <p><b>Universidad Internacional de La Rioja (UNIR)</b></p>
  <p><b>Actividad Grupal — Busqueda Heuristica</b></p>
  <p style="color:#a0c4ff; font-size:0.9em;">Lenguaje: Python 3 · Algoritmo: A* con distancia Manhattan</p>
</div>

## Indice

1. Descripcion del Problema
2. Analisis del Estado Inicial y Final
3. Fundamentos Teoricos — Algoritmo A*
4. Arquitectura del Codigo
5. Implementacion Completa
6. Ejecucion y Plan Generado
7. Visualizacion del Recorrido
8. Pruebas Realizadas
9. Dificultades Encontradas


## 1. Descripcion del Problema

La empresa **Amazon** requiere que un robot autonomo ordene el inventario de su almacen.
El robot debe mover **tres inventarios** (M1, M2 y M3) desde sus posiciones iniciales
hasta posiciones objetivos predefinidas, navegando por una cuadricula **4x4** que contiene paredes.

### Acciones disponibles del robot
| Accion | Descripcion |
|--------|-------------|
| `moverR` | Desplazarse a una celda adyacente (arriba, abajo, izquierda, derecha) |
| `cargarR` | Recoger un inventario en la posicion actual |
| `descargarR` | Depositar el inventario cargado en la posicion actual |

### Restricciones
- El robot **solo** puede moverse horizontal o verticalmente (sin diagonal).
- Las celdas marcadas con `#` son **paredes** infranqueables.
- El coste de cada movimiento es **uniforme** (g = 1 por paso).
- La heuristica usada es la **distancia Manhattan**.


## 2. Analisis del Estado Inicial y Final

A continuacion se definen los dos estados del problema como matrices 4x4.

In [ ]:
# Definicion de los estados
INITIAL_STATE = [
    ["M1", "#",  "",   "M3"],   # fila 0
    ["M2", "#",  "",   ""  ],   # fila 1
    ["",   "",   "R",  ""  ],   # fila 2
    ["",   "",   "",   ""  ],   # fila 3
]

FINAL_STATE = [
    ["",  "",   "",   "" ],     # fila 0
    ["",  "",   "",   "" ],     # fila 1
    ["",  "",   "",   "" ],     # fila 2
    ["",  "M3", "M2", "M1"],   # fila 3
]

def print_grid(grid, title=''):
    cols = len(grid[0])
    sep = '+' + ('+'.join(['------'] * cols)) + '+'
    print(f'\n{title}')
    print('-' * 30)
    for r, row in enumerate(grid):
        print(sep)
        cells_str = ' | '.join(f'{c:^4}' for c in row)
        print(f'| {cells_str} |  <- fila {r}')
    print(sep)
    header = '    ' + '   '.join(f'c{c}' for c in range(cols))
    print(header)

print_grid(INITIAL_STATE, 'ESTADO INICIAL')
print()
print_grid(FINAL_STATE, 'ESTADO OBJETIVO')


### Leyenda del mapa

| Simbolo | Significado | Posicion inicial | Destino |
|---------|-------------|-----------------|--------|
| `R` | Robot | [2, 2] | — |
| `M1` | Inventario 1 | [0, 0] | [3, 3] |
| `M2` | Inventario 2 | [1, 0] | [3, 2] |
| `M3` | Inventario 3 | [0, 3] | [3, 1] |
| `#` | Pared | [0,1] y [1,1] | — |

> **Nota:** La pared en columna 1 (filas 0 y 1) obliga al robot a usar
> la columna 0 o las columnas 2-3 para acceder a M1 y M2.


## 3. Fundamentos Teoricos — Algoritmo A*

### Que es A*?
A* (Hart, Nilsson & Raphael, 1968) es un algoritmo de busqueda informada que combina:
- **Busqueda de coste uniforme** (Dijkstra): garantiza el camino de menor coste g(n).
- **Busqueda voraz** (Greedy Best-First): usa una heuristica h(n) para orientar la exploracion.

La funcion de evaluacion de cada nodo es:

$$f(n) = g(n) + h(n)$$

Donde:
- **g(n)**: coste real acumulado desde el inicio hasta el nodo n.
- **h(n)**: estimacion del coste desde n hasta el objetivo (heuristica).
- **f(n)**: coste total estimado del camino optimo que pasa por n.

### Distancia Manhattan como heuristica

$$h(n) = |fila_n - fila_{obj}| + |col_n - col_{obj}|$$

Esta heuristica es **admisible** (nunca sobreestima el coste real) y **consistente**
(satisface la desigualdad triangular), lo que garantiza que A* encuentre siempre
el camino **optimo**.

### Estructura de las listas
- **Lista abierta (open list):** nodos descubiertos pero aun no expandidos. Implementada como un **min-heap** ordenado por f(n).
- **Lista cerrada (closed list):** nodos ya expandidos. Se usa para evitar re-explorar nodos visitados.

### Pseudocodigo de A*
```
OPEN  = {nodo_inicio}
CLOSED = {}

mientras OPEN no este vacia:
    n = extraer nodo de OPEN con menor f(n)
    si n == objetivo:
        devolver camino reconstruido
    agregar n a CLOSED
    para cada vecino v de n:
        si v en CLOSED: continuar
        g_nuevo = g(n) + coste(n, v)
        h_nuevo = distancia_manhattan(v, objetivo)
        f_nuevo = g_nuevo + h_nuevo
        agregar v a OPEN con prioridad f_nuevo
```


In [ ]:
# Demostracion de la distancia Manhattan
def manhattan_distance(origin, destination):
    return abs(origin[0]-destination[0]) + abs(origin[1]-destination[1])

test_cases = [
    ((2,2), (0,0), 'Robot -> M1 (inicio)'),
    ((2,2), (1,0), 'Robot -> M2 (inicio)'),
    ((2,2), (0,3), 'Robot -> M3 (inicio)'),
    ((0,0), (3,3), 'M1 inicio -> destino'),
    ((1,0), (3,2), 'M2 inicio -> destino'),
    ((0,3), (3,1), 'M3 inicio -> destino'),
]

print('Calculo de h(n) para pares de posiciones:')
print(f'{"Par":<30}  h(n)')
print('-' * 38)
for origin, dest, label in test_cases:
    h = manhattan_distance(origin, dest)
    print(f'{label:<35}  {h}')

print()
print('La heuristica es ADMISIBLE y CONSISTENTE.')
print('Garantiza que A* encuentra siempre el camino optimo.')


## 4. Arquitectura del Codigo

El proyecto esta organizado en modulos independientes:

```
proyecto_python/
  src/
    action.py      -> Enum con las acciones del robot
    direction.py   -> Enum con las direcciones (UP, DOWN, LEFT, RIGHT)
    exceptions.py  -> Excepciones personalizadas
    activity.py    -> Dataclass que representa un paso del plan
    app.py         -> Clase App con el algoritmo A*
    main.py        -> Punto de entrada
```

### Decisiones de diseno clave

| Decision | Justificacion |
|----------|---------------|
| `heapq` (min-heap) para open list | O(log n) por insercion/extraccion |
| `set` para nodos visitados | Busqueda O(1) vs O(n) de lista |
| Path completo en cada entrada del heap | Evita tabla de padres separada |
| Counter de desempate | Comportamiento deterministico cuando f(n) es igual |
| `sorted()` para procesar inventarios | Orden reproducible M1 -> M2 -> M3 |


## 5. Implementacion Completa

Codigo fuente de cada modulo del proyecto.

### 5.1 `action.py` — Acciones del robot

In [ ]:
from enum import Enum

class Action(Enum):
    INITIAL = 'inicialR'
    FINAL   = 'finalR'
    PICKUP  = 'cargarR'
    DROP    = 'descargarR'
    MOVE    = 'moverR'

for a in Action:
    print(f'  {a.name:10} -> {a.value}')


### 5.2 `direction.py` — Direcciones de movimiento

In [ ]:
from enum import Enum

class Direction(Enum):
    UP    = 'UP'
    DOWN  = 'DOWN'
    LEFT  = 'LEFT'
    RIGHT = 'RIGHT'

print('Orden de exploracion:', [d.value for d in Direction])


### 5.3 `exceptions.py` — Excepciones personalizadas

In [ ]:
class OutOfLimitException(Exception):
    pass

class NotAbleToMoveException(Exception):
    pass

print('Excepciones: OutOfLimitException, NotAbleToMoveException')


### 5.4 `activity.py` — Paso del plan

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, Tuple

@dataclass
class Activity:
    position: Tuple[int, int]
    action: Action
    detail: Optional[str] = field(default=None)

    def __str__(self):
        row, col = self.position
        if self.detail:
            return f'{self.action.value} ({self.detail},{row},{col})'
        return f'{self.action.value} ({row},{col})'

print(Activity((2,2), Action.INITIAL))
print(Activity((0,0), Action.PICKUP, 'M1'))


### 5.5 `app.py` — Clase App con el algoritmo A*

In [ ]:
import heapq
from typing import Dict, List, Optional, Set, Tuple

UNIFORM_COST = 1

class App:
    def __init__(self):
        self.open_list   = []
        self.closed_list = []
        self.package_positions    = {}
        self.package_destinations = {}
        self.robot_position = None
        self.obstacles = []
        self.plan = []

    def initialize(self):
        for r, row in enumerate(INITIAL_STATE):
            for c, cell in enumerate(row):
                if cell.startswith('M'):   self.package_positions[cell] = (r, c)
                elif cell == 'R':          self.robot_position = (r, c)
                elif cell == '#':          self.obstacles.append((r, c))
        for r, row in enumerate(FINAL_STATE):
            for c, cell in enumerate(row):
                if cell.startswith('M'):   self.package_destinations[cell] = (r, c)
        self.closed_list.append(self.robot_position)
        self.plan.append(Activity(self.robot_position, Action.INITIAL))

    def finalize(self):
        self.plan.append(Activity(self.robot_position, Action.FINAL))

    def move_robot(self, package_name, target, action):
        start = self.robot_position
        if start == target:
            self.plan.append(Activity(self.robot_position, action, package_name))
            return
        counter = 0
        heap = []
        heapq.heappush(heap, (0, 0, counter, start, [start]))
        visited = set()
        while heap:
            f, g, _, current, path = heapq.heappop(heap)
            if current in visited:
                continue
            visited.add(current)
            self.closed_list = list(visited)
            self.open_list = [item[3] for item in heap if item[3] not in visited]
            if current == target:
                for pos in path[1:]:
                    self.plan.append(Activity(pos, Action.MOVE))
                self.plan.append(Activity(current, action, package_name))
                self.robot_position = current
                self.closed_list.clear()
                return
            for neighbor in self._get_neighbors_for(current, visited):
                new_g = g + UNIFORM_COST
                h = abs(neighbor[0]-target[0]) + abs(neighbor[1]-target[1])
                counter += 1
                heapq.heappush(heap, (new_g+h, new_g, counter, neighbor, path+[neighbor]))

    def _get_neighbors_for(self, pos, visited):
        rows, cols = len(INITIAL_STATE), len(INITIAL_STATE[0])
        r, c = pos
        result = []
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < rows and 0 <= nc < cols:
                nxt = (nr, nc)
                if nxt not in self.obstacles and nxt not in visited:
                    result.append(nxt)
        return result

print('Clase App definida correctamente.')


## 6. Ejecucion y Plan Generado

Se ejecuta el programa completo y se muestra la secuencia de acciones generada por A*.

In [ ]:
robot = App()
robot.initialize()

print('=' * 55)
print('  CONFIGURACION INICIAL')
print('=' * 55)
print(f'  Posicion robot      : {robot.robot_position}')
print(f'  Posiciones iniciales: {robot.package_positions}')
print(f'  Posiciones objetivo : {robot.package_destinations}')
print(f'  Obstaculos          : {robot.obstacles}')
print()

for package_name in sorted(robot.package_positions.keys()):
    robot.move_robot(package_name, robot.package_positions[package_name], Action.PICKUP)
    robot.move_robot(package_name, robot.package_destinations[package_name], Action.DROP)

robot.finalize()

print('=' * 55)
print('  PLAN DE EJECUCION GENERADO POR A*')
print('=' * 55)
for i, act in enumerate(robot.plan, 1):
    marker = '>>>' if act.action in (Action.PICKUP, Action.DROP) else '   '
    print(f'  Paso {i:02d} {marker} {act}')
print()
print(f'  Total de pasos: {len(robot.plan)}')


## 7. Visualizacion del Recorrido

Se genera una visualizacion grafica del estado inicial, la trayectoria del robot y el estado final.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print('matplotlib no instalado. Ejecutar: pip install matplotlib')


In [ ]:
def visualize_astar(plan, initial, final, obstacles):
    if not HAS_MPL:
        print('matplotlib requerido para esta visualizacion.')
        return

    ROWS, COLS = 4, 4
    C_EMPTY  = '#f0f4ff'
    C_WALL   = '#2c3e50'
    C_ROBOT  = '#3498db'
    C_INV    = '#e74c3c'
    C_PATH   = '#f39c12'
    C_PICKUP = '#9b59b6'
    C_DROP   = '#1abc9c'
    C_DEST   = '#2ecc71'

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor('#1a1a2e')

    def draw_grid(ax, grid, title, path=None, highlights=None):
        ax.set_facecolor('#1a1a2e')
        ax.set_title(title, color='white', fontsize=13, pad=12, fontweight='bold')
        ax.set_xlim(-0.5, COLS-0.5)
        ax.set_ylim(-0.5, ROWS-0.5)
        ax.set_aspect('equal')
        ax.invert_yaxis()
        for r in range(ROWS):
            for c in range(COLS):
                v = grid[r][c]
                color = C_WALL if v=='#' else C_ROBOT if v=='R' else C_INV if v.startswith('M') else C_EMPTY
                rect = plt.Rectangle((c-0.5,r-0.5),1,1,linewidth=1.5,edgecolor='#34495e',facecolor=color,zorder=1)
                ax.add_patch(rect)
                if v:
                    ax.text(c,r,v,ha='center',va='center',fontsize=11,fontweight='bold',
                            color='white' if v in ('#','R') else '#2c3e50',zorder=3)
        if path:
            xs=[p[1] for p in path]; ys=[p[0] for p in path]
            ax.plot(xs,ys,'o-',color=C_PATH,linewidth=2,markersize=6,alpha=0.7,zorder=2)
        if highlights:
            for pos, col, _ in highlights:
                circ=plt.Circle((pos[1],pos[0]),0.3,color=col,alpha=0.75,zorder=4)
                ax.add_patch(circ)
        ax.set_xticks(range(COLS)); ax.set_yticks(range(ROWS))
        ax.set_xticklabels([f'c{i}' for i in range(COLS)],color='#aaa')
        ax.set_yticklabels([f'f{i}' for i in range(ROWS)],color='#aaa')
        for s in ax.spines.values(): s.set_edgecolor('#34495e')

    full_path=[a.position for a in plan if a.action not in (Action.FINAL,)]
    pickups=[(a.position,C_PICKUP,a.detail) for a in plan if a.action==Action.PICKUP]
    drops=[(a.position,C_DROP,a.detail) for a in plan if a.action==Action.DROP]

    draw_grid(axes[0], initial, 'Estado Inicial')
    draw_grid(axes[1], initial, 'Trayectoria del Robot', path=full_path, highlights=pickups+drops)

    final_grid=[[''] * COLS for _ in range(ROWS)]
    for r,row in enumerate(final):
        for c,v in enumerate(row): final_grid[r][c]=v
    draw_grid(axes[2], final_grid, 'Estado Objetivo Alcanzado')

    legend=[mpatches.Patch(color=c,label=l) for c,l in [
        (C_INV,'Inventario (origen)'),(C_WALL,'Pared'),(C_ROBOT,'Robot'),
        (C_PATH,'Trayectoria'),(C_PICKUP,'PICKUP'),(C_DROP,'DROP')]]
    fig.legend(handles=legend,loc='lower center',ncol=3,facecolor='#16213e',
               edgecolor='#34495e',labelcolor='white',fontsize=9,bbox_to_anchor=(0.5,-0.02))
    plt.suptitle('Robot Amazon — Busqueda A*',color='white',fontsize=15,fontweight='bold',y=1.02)
    plt.tight_layout()
    plt.savefig('plan_astar.png',dpi=150,bbox_inches='tight',facecolor='#1a1a2e')
    plt.show()
    print('Grafico guardado como plan_astar.png')

visualize_astar(robot.plan, INITIAL_STATE, FINAL_STATE, robot.obstacles)


## 8. Pruebas Realizadas

Se realizaron pruebas unitarias y de integracion para verificar la corrección del algoritmo.

In [ ]:
print('=' * 60)
print('  PRUEBA 1: Admisibilidad de la heuristica Manhattan')
print('=' * 60)

test_cases = [
    ((2,2), (0,0), 'Robot -> M1'),
    ((2,2), (1,0), 'Robot -> M2'),
    ((2,2), (0,3), 'Robot -> M3'),
    ((0,0), (3,3), 'M1 inicio -> destino'),
    ((1,0), (3,2), 'M2 inicio -> destino'),
    ((0,3), (3,1), 'M3 inicio -> destino'),
]

for origin, dest, label in test_cases:
    h = abs(origin[0]-dest[0]) + abs(origin[1]-dest[1])
    print(f'  {label:<35} h = {h}')
print()
print('  -> Heuristica ADMISIBLE (todos los valores >= 0) [OK]')


In [ ]:
print('=' * 60)
print('  PRUEBA 2: Robot ya en la posicion objetivo')
print('=' * 60)

r2 = App()
r2.initialize()
r2.robot_position = (0, 0)
before = len(r2.plan)
r2.move_robot('M1', (0, 0), Action.PICKUP)
after = len(r2.plan)
print(f'  Pasos antes: {before}  Pasos despues: {after}')
print(f'  Accion agregada: {r2.plan[-1]}')
assert after == before + 1, 'Error: deberia agregar solo 1 accion'
print('  -> Sin movimiento innecesario [OK]')


In [ ]:
print('=' * 60)
print('  PRUEBA 3: Todos los inventarios llegan a su destino')
print('=' * 60)

r3 = App()
r3.initialize()
for pname in sorted(r3.package_positions.keys()):
    r3.move_robot(pname, r3.package_positions[pname], Action.PICKUP)
    r3.move_robot(pname, r3.package_destinations[pname], Action.DROP)
r3.finalize()

drops = [a for a in r3.plan if a.action == Action.DROP]
expected = {'M1':(3,3), 'M2':(3,2), 'M3':(3,1)}
all_ok = True
for d in drops:
    ok = d.position == expected[d.detail]
    status = 'OK' if ok else 'FALLO'
    print(f'  {d.detail} DROP en {d.position} == esperado {expected[d.detail]} [{status}]')
    if not ok: all_ok = False
print()
print('  -> Todos los destinos alcanzados:', 'SI [OK]' if all_ok else 'NO [FALLO]')


In [ ]:
print('=' * 60)
print('  PRUEBA 4: Conteo de pasos por sub-tarea')
print('=' * 60)

current = []
subtasks = []
for act in r3.plan:
    current.append(act)
    if act.action == Action.PICKUP:
        subtasks.append((f'Ir a {act.detail} (PICKUP)', len(current)))
        current = []
    elif act.action == Action.DROP:
        subtasks.append((f'Llevar {act.detail} a destino (DROP)', len(current)))
        current = []

for name, steps in subtasks:
    print(f'  {name:<42} {steps:3d} pasos')
print(f'  {"TOTAL (incluyendo INICIAL y FINAL)":<42} {len(r3.plan):3d} pasos')


## 9. Dificultades Encontradas

### 1. Diferencia entre A* greedy y A* completo

La implementacion de referencia en Java usa una aproximacion *greedy paso a paso*: en cada iteracion
elige el unico mejor vecino sin mantener una frontera global. Esto puede atrapar al robot sin
vecinos disponibles (todos en la lista cerrada). Se implemento A* completo con `heapq` que garantiza
encontrar el camino optimo.

### 2. Desempate entre nodos con igual f(n)

Cuando varios nodos tienen el mismo f(n), Python necesita un criterio de desempate en la tupla del heap.
Se introduce un `counter` entero que garantiza orden FIFO determinístico.

### 3. Coordenadas (fila, columna) vs (x, y)

Java usa `Point(x, y)` donde x puede ser columna, causando ambiguedad. En Python se adopto
la convencion `(fila, columna)` de forma consistente en todo el codigo.

### 4. Orden de procesamiento de inventarios

Java usa `HashMap` que no garantiza orden, mientras que Python con `sorted()` procesa
M1 -> M2 -> M3. Esto expuso el bug de la implementacion greedy con M2 que el azar del
HashMap ocultaba en Java.

### 5. Reconstruccion del camino optimo

En lugar de una tabla de padres separada, se incluye el path completo dentro de cada entrada del heap.
Simplifica el codigo a costa de mayor uso de memoria — aceptable para el grid 4x4 de este problema.


In [ ]:
print('=' * 60)
print('  RESUMEN FINAL')
print('=' * 60)
print()
print('  Problema   : Robot Amazon mueve 3 inventarios en grid 4x4')
print('  Algoritmo  : A* con heuristica distancia Manhattan')
print('  Heuristica : Admisible y consistente -> solucion OPTIMA')
print('  Coste g(n) : Uniforme (1 por movimiento)')
print()
print('  Resultados:')
print(f'    Pasos totales en el plan : {len(r3.plan)}')
print(f'    Sub-tareas completadas   : 6 (3 PICKUP + 3 DROP)')
print(f'    Inventarios entregados   : M1->[3,3] | M2->[3,2] | M3->[3,1]')
print()
print('  A* garantiza el camino optimo para cada sub-tarea,')
print('  evitando obstaculos y manteniendose dentro del grid.')
